# 03 — Embeddings Intuition + Tiny Semantic Search

**Big idea:** embeddings turn text into vectors where similar meaning is closer.

## Why this comes after simple RAG

In Notebook 02, you learned RAG in action: retrieve context → augment prompt → generate answer.

But how does "retrieval" actually work? How do we find the *relevant* chunks?

**This notebook:** Dive under the hood to understand embeddings and semantic search — the mechanism that powers the **R** (Retrieval) step in RAG.

## What you will do

**Goal:** Understand how embeddings and similarity ranking power retrieval.

1. Create a tiny class-related text corpus
2. Convert text → vectors (embeddings)
3. Compute similarity between vectors (cosine similarity)
4. Build a mini semantic search

By the end, you'll see: **query → embedding → compare → rank → best matches** ← This is how retrieval works.

In [ ]:
# Install once if needed:
# !pip install -q gpt4all numpy

# Import Embed4All for text-to-vector conversion
# numpy for vector math and similarity calculations
import numpy as np
from gpt4all import Embed4All

In [ ]:
# Create a small corpus of texts related to design class activities
# Mix design tasks, non-design content, and edge cases to test retrieval
texts = [
    "Use AI to brainstorm poster concepts.",
    "Generate naming ideas for a student exhibition.",
    "How to cook pasta with tomato sauce.",
    "Design critique: improve hierarchy and contrast.",
    "Write a short artist statement draft.",
    "Train schedule from Zurich to Lucerne.",
    "Create 5 prompts for visual moodboard exploration.",
    "Explain color harmony in simple terms.",
    "Python function for sorting a list.",
    "Draft interview questions for a creative project."
]
print(f"Loaded {len(texts)} texts")

## Step 1 — Embed the corpus

Each text becomes a numeric vector. The model chooses the vector dimension.

In [ ]:
# Initialize embedder and convert all texts to embedding vectors
embedder = Embed4All()

# Each text becomes a dense numeric vector (dimension determined by the model)
text_embeddings = np.array([embedder.embed(t) for t in texts])

# Print the shape of the embeddings array to confirm dimensions
print("Embeddings shape:", text_embeddings.shape)

In [ ]:
# Pick one text and its embedding to inspect
idx = 0

# Peek inside one embedding to see what they look like
print("Selected text:", texts[idx])
print("Its embedding (first 10 values):", text_embeddings[idx][:10])
print("Full embedding length:", len(text_embeddings[idx]))
print(f"Matrix shape: {text_embeddings.shape[0]} texts × {text_embeddings.shape[1]} dimensions")

### Understanding the shape

When you print `Embeddings shape`, you see something like **(10, 384)** or **(10, 768)**.

**What this means:**

- **First number (10)** = number of texts you embedded
  - You have 10 texts in the corpus
  - Each one becomes ONE vector

- **Second number (384 or 768)** = embedding dimension (or "model size")
  - Each text is represented as a vector with 384 (or 768) numbers
  - These numbers capture the text's "meaning" in mathematical space
  - Larger dimensions = more expressive but heavier

**Visual example:**

```
Text 1: "Use AI to brainstorm poster concepts."
        → [0.23, -0.15, 0.89, ..., 0.42]  (384 numbers)

Text 2: "Generate naming ideas for a student exhibition."
        → [0.25, -0.12, 0.91, ..., 0.44]  (384 numbers)

Text 3: "How to cook pasta with tomato sauce."
        → [-0.10, 0.67, 0.03, ..., -0.55]  (384 numbers)
```

Notice: Text 1 and 2 have *similar* numbers (closer meaning). Text 3 is very different.

**This is what cosine similarity will measure:** How close are these vectors to each other?

### From numbers to similarity

Embeddings are just vectors. The real question: **How do we measure if two vectors are similar?**

**Answer: Cosine similarity** — a mathematical measure of how aligned two vectors are.

Let's compute it for 3 texts: 2 design-related + 1 cooking. We should see high similarity for the design pair, low for design vs cooking.

In [ ]:
# Define a cosine similarity function
def quick_cosine(v1, v2):
    """Compute cosine similarity between two vectors"""
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

# Select 3 texts: 2 design-related, 1 unrelated
text_a = texts[0]  # "Use AI to brainstorm poster concepts."
text_b = texts[1]  # "Generate naming ideas for a student exhibition."
text_c = texts[2]  # "How to cook pasta with tomato sauce."

# Compute similarities between their embeddings
sim_a_b = quick_cosine(text_embeddings[0], text_embeddings[1])  # Design vs Design
sim_a_c = quick_cosine(text_embeddings[0], text_embeddings[2])  # Design vs Cooking
sim_b_c = quick_cosine(text_embeddings[1], text_embeddings[2])  # Design vs Cooking

# Display results
print("=== COSINE SIMILARITY: The Real Proof ===\n")
print(f"Text A: {text_a}")
print(f"Text B: {text_b}")
print(f"Text C: {text_c}")
print()
print(f"A vs B (design + design):")
print(f"  Cosine Similarity = {sim_a_b:.4f} ← HIGH (similar meaning)")
print()
print(f"A vs C (design + cooking):")
print(f"  Cosine Similarity = {sim_a_c:.4f} ← LOW (different meaning)")
print()
print(f"B vs C (design + cooking):")
print(f"  Cosine Similarity = {sim_b_c:.4f} ← LOW (different meaning)")
print()
print(f"✓ Proof: Similar texts have HIGH cosine similarity scores")
print(f"✓ Different texts have LOW cosine similarity scores")

## Step 2 — Similarity function

We define our cosine similarity function 

In [ ]:
# Compute cosine similarity between two vectors
# Compares direction (meaning) not magnitude (length)
# Returns: 1.0 = identical direction, 0.0 = perpendicular, -1.0 = opposite
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## Step 3 — Run tiny semantic search


In [ ]:
# Run semantic search: find texts most similar to a query in meaning
# Notice: query words don't need to match text words exactly
query = "I need ideas for AI-assisted design prompts"
# Embed the query
q_emb = embedder.embed(query)

# Calculate similarity between query and all texts
scores = [cosine_similarity(q_emb, emb) for emb in text_embeddings]
# Sort text indices by score (descending)
ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)

print("Query:", query)
print("\nTop 3 matches:")
# Display highest-scoring matches with their scores
for idx, score in ranked[:3]:
    print(f"- ({score:.3f}) {texts[idx]}")

### What to observe

- Are the top matches semantically related, even if wording differs?
- Does one irrelevant text still appear in top 3? Why might that happen?

## Mini challenge

Change the query twice:
- one creative query
- one technical query

Did the top results shift logically?

## Why this matters for your class

In Notebook 02, you saw: **retrieve context → augment prompt → generate answer**

This notebook shows you *how* the first step (retrieve) actually works:

- **Embeddings** convert text into vectors where similar meanings cluster together
- **Cosine similarity** ranks which chunks are closest in meaning to the user's question
- That ranking is what lets you pull the *right* facts into the prompt

Without this retrieval mechanism, RAG wouldn't work. With it, you can make LLMs answer from your data instead of hallucinating.

## Reflection (write 3–5 lines)

- Which match surprised you?
- Where could you use this in your own project workflow?